# Colab 28 — ESM2 vs SNN, refreshed: AUROC + MAP@k + 3-part runtime (current architecture)

Refresh of **colab18** on the **current** setup. The SNN was already `AdaptiveAvgPool K=16`; what
changes here:

1. **Eval = colab24e protocol.** Genuinely **per-representation pools** + **feed-matched** high pairs
   from the raw pair files, every similarity judged by **recomputed `normLev` on current strings**
   (the same metric the full-pool exact-Levenshtein oracle uses). Replaces the old AA-centred
   `cath_eval.csv.gz`.
2. **Both quality metrics, side by side:** **AUROC** (pairwise: is-high vs the rest) **and MAP@10**
   (full-pool, rank-aware, vs the exact-Levenshtein neighbour set) — for **SNN-free** and **ESM2-free**,
   against a **length-only baseline**. (MAP's ceiling is 1.0 by construction; recall@k is shown too as
   the crowding-bounded companion.)
3. **Runtime in three parts:** **construction** (one-time: SNN data-gen+train+pool-encode vs ESM2
   load+pool-encode; exact Lev = 0), **inference** (per query: encode the query + search the pool), and
   **all-together** (end-to-end for Q queries = construction + Q×inference — the amortization curve, vs
   exact Levenshtein which has no construction but pays full cost every query).

**Framing (unchanged).** ESM2 is the **data-dependent foil**, not a competitor: a protein LM trained
on ~tens of M UniRef sequences for a *biological* objective — strong "lots of data, wrong objective."
ESM2-35M is the **conservative** choice (larger only helps it on AA). **No biological claim** —
relevance is the exact-Levenshtein high-string-similarity set. AA high = n_pos 5 (underpowered → read
SS/3Di). The honest headline from colab18 stands and is re-tested here: SNN wins **pairwise + efficiency**;
at **retrieval** ESM2-free ties/edges the SNN — so the SNN case is **efficiency + zero-shot**, not a
retrieval win.


## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
CACHE = '/content/bench_cache'; os.makedirs(CACHE, exist_ok=True)
import os
for f in ['cath_s20_pairs_sample.csv.gz', 'cath_s20_pairs_high.csv.gz',
          'cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch rapidfuzz scikit-learn scipy transformers matplotlib --quiet

In [ ]:
import time, json, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLev
from rapidfuzz.process import cdist as rf_cdist
SEED = 42; torch.manual_seed(SEED); np.random.seed(SEED); rng = np.random.default_rng(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 2. Constants + helpers + SNN architecture (current colab24e recipe)

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'; SS_ALPHABET = 'HLS'
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}; PAD_IDX = 20; VOCAB = 21
MIN_LEN, MAX_LEN = 50, 200; N_TRAIN, EPOCHS, BS, K = 30000, 30, 128, 16
BAND_LOW_AA, BAND_LOW_SS, BAND_HIGH = 0.30, 0.56, 0.70
AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s); is_ss = lambda s: all(c in SS_SET for c in s)
def norm_lev(a, b):
    L = max(len(a), len(b)); return 1.0 if L == 0 else 1.0 - RFLev.distance(a, b) / L
def encode_pad(seq):
    idx = [CHAR_TO_IDX[c] for c in seq][:MAX_LEN]; idx += [PAD_IDX]*(MAX_LEN-len(idx))
    return torch.tensor(idx, dtype=torch.long)
def perturb(seq, k, abc):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= MAX_LEN: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub': i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins': i = rng.integers(0, len(s)+1); s.insert(i, rng.choice(abc))
        else: i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)
def rand_seq(abc): L = int(rng.integers(MIN_LEN, MAX_LEN+1)); return ''.join(rng.choice(list(abc), size=L))
def bin_idx(x, band_low): return 0 if x < band_low else (1 if x < BAND_HIGH else 2)

In [ ]:
class Enc(nn.Module):
    def __init__(s):
        super().__init__(); s.emb = nn.Embedding(VOCAB, 32, padding_idx=PAD_IDX)
        s.c1 = nn.Conv1d(32, 32, 3, padding=1); s.c2 = nn.Conv1d(32, 64, 3, padding=1)
        s.pool = nn.AdaptiveAvgPool1d(K); s.fc = nn.Linear(64*K, 128)
    def forward(s, x):
        m = (x != PAD_IDX).float(); e = s.emb(x).permute(0, 2, 1)
        h = F.relu(s.c1(e)); h = F.relu(s.c2(h)); h = h * m.unsqueeze(1)
        return F.normalize(s.fc(s.pool(h).flatten(1)), p=2, dim=1)
class Clf(nn.Module):
    def __init__(s):
        super().__init__(); s.encoder = Enc()
        s.head = nn.Sequential(nn.Linear(128, 64), nn.LeakyReLU(0.01), nn.Linear(64, 3))
    def forward(s, a, b): return s.head(torch.abs(s.encoder(a) - s.encoder(b)))
class DS(Dataset):
    def __init__(s, pp, bl): s.p = pp; s.bl = bl
    def __len__(s): return len(s.p)
    def __getitem__(s, i): a, b, l = s.p[i]; return encode_pad(a), encode_pad(b), bin_idx(l, s.bl)

def train_snn(alphabet, band_low, label):
    t0 = time.perf_counter(); pairs = []
    while len(pairs) < N_TRAIN:
        sd = rand_seq(alphabet); L = len(sd); t = float(rng.uniform(0, 1)); k = max(0, int(round((1-t)*L)))
        o = perturb(sd, k, alphabet)
        if 1 <= len(o) <= MAX_LEN: pairs.append((sd, o, norm_lev(sd, o)))
    datagen_s = time.perf_counter() - t0
    dl = DataLoader(DS(pairs, band_low), batch_size=BS, shuffle=True)
    model = Clf().to(device); opt = torch.optim.Adam(model.parameters(), 1e-3); crit = nn.CrossEntropyLoss()
    model.train(); t0 = time.perf_counter()
    for ep in range(1, EPOCHS+1):
        tot = nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = crit(model(a, b), y); opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item(); nb += 1
        if ep % 10 == 0 or ep == 1: print(f'  [{label}] epoch {ep}/{EPOCHS} CE {tot/nb:.4f}')
    if device.type == 'cuda': torch.cuda.synchronize()
    train_s = time.perf_counter() - t0
    model.eval(); return model, datagen_s, train_s

print('training AA + SS SNN encoders (timing captured for the construction comparison)...')
model_aa, AA_DATAGEN_S, AA_TRAIN_S = train_snn(AA_ALPHABET, BAND_LOW_AA, 'AA')
model_ss, SS_DATAGEN_S, SS_TRAIN_S = train_snn(SS_ALPHABET, BAND_LOW_SS, 'SS')
snn_params = sum(p.numel() for p in model_aa.parameters())
print(f'SNN params = {snn_params:,} | AA train one-time = {AA_DATAGEN_S+AA_TRAIN_S:.1f}s')

## 3. Per-representation pools + feed-matched eval (colab24e protocol)

In [ ]:
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')
RESCUED = {'4z0mC02', '3qkaE02'}
def _valid(seq, isstd, d):
    return (isinstance(seq, str) and isstd(seq) and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))
id_to_aa  = {d: s for d, s in zip(raw['domain_id'], raw['aa_seq']) if _valid(s, is_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'], raw['ss_seq']) if _valid(s, is_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_aa, d)}
LOOK = {'AA': id_to_aa, 'SS': id_to_ss, '3Di': id_to_3di}
POOL_IDS = {f: list(LOOK[f]) for f in LOOK}; POOL_SET = {f: set(POOL_IDS[f]) for f in LOOK}
FEEDS = ['AA', 'SS', '3Di']; SNN_BY_FEED = {'AA': model_aa, 'SS': model_ss, '3Di': model_aa}

PAIR_COLS = ['domain_a', 'domain_b', 'ss_score', 'aa_score', 'src4', 'src5', 'src_flag']
samp = pd.read_csv(f'{DATA_DIR}/cath_s20_pairs_sample.csv.gz', header=None, names=PAIR_COLS)
high = pd.read_csv(f'{DATA_DIR}/cath_s20_pairs_high.csv.gz', header=None, names=PAIR_COLS)
src = pd.concat([samp, high], ignore_index=True); src = src[src.domain_a != src.domain_b]
src['pk'] = [frozenset((a, b)) for a, b in zip(src.domain_a, src.domain_b)]
src = src.drop_duplicates('pk').reset_index(drop=True)
def build_feed_eval(feed):
    inp = POOL_SET[feed]; lk = LOOK[feed]
    sub = src[src.domain_a.isin(inp) & src.domain_b.isin(inp)]
    a = sub.domain_a.values; b = sub.domain_b.values
    cur = np.array([norm_lev(lk[x], lk[y]) for x, y in zip(a, b)])
    return pd.DataFrame({'domain_a': a, 'domain_b': b, 'current_normlev': cur,
                         'is_high': (cur >= BAND_HIGH).astype(int)})
EVAL = {f: build_feed_eval(f) for f in FEEDS}
def feed_high(feed): return EVAL[feed][EVAL[feed].is_high == 1]
QUERIES = {f: sorted(set(feed_high(f).domain_a) | set(feed_high(f).domain_b)) for f in FEEDS}
for f in FEEDS:
    print(f'{f:<4} pool={len(POOL_IDS[f]):>6}  eligible={len(EVAL[f]):>6}  high={int(feed_high(f).shape[0]):>4}  queries={len(QUERIES[f]):>4}')

## 4. ESM2 (frozen, mean-pooled) — the data-dependent foil

In [ ]:
from transformers import AutoTokenizer, AutoModel
ESM2_MODEL = 'facebook/esm2_t12_35M_UR50D'   # conservative; swap 'facebook/esm2_t33_650M_UR50D' on GPU
t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(ESM2_MODEL); esm = AutoModel.from_pretrained(ESM2_MODEL).to(device).eval()
ESM_LOAD_S = time.perf_counter() - t0
esm_params = sum(p.numel() for p in esm.parameters())
ESM_DIM = esm.config.hidden_size
print(f'ESM2 = {ESM2_MODEL} | params={esm_params:,} | dim={ESM_DIM} | load {ESM_LOAD_S:.1f}s | '
      f'param ratio ESM2/SNN = {esm_params/snn_params:.0f}x')

@torch.no_grad()
def esm_embed(seqs, bs=32):
    order = np.argsort([len(s) for s in seqs]); out = [None]*len(seqs)
    for i in range(0, len(order), bs):
        idx = order[i:i+bs]; batch = [seqs[j] for j in idx]
        enc = tok(batch, return_tensors='pt', padding=True, add_special_tokens=True).to(device)
        h = esm(**enc).last_hidden_state
        mask = enc['attention_mask'].clone(); mask[:, 0] = 0
        for r, l in enumerate(enc['attention_mask'].sum(1)): mask[r, l-1] = 0   # drop cls + eos
        m = mask.unsqueeze(-1).float(); e = (h*m).sum(1)/m.sum(1).clamp(min=1)
        e = F.normalize(e, dim=1).cpu().numpy()
        for kk, j in enumerate(idx): out[j] = e[kk]
    return np.stack(out)

## 5. Pool embeddings (SNN + ESM2), with caching

In [ ]:
@torch.no_grad()
def snn_pool(feed):
    look = LOOK[feed]; ids = POOL_IDS[feed]; model = SNN_BY_FEED[feed]
    out = np.zeros((len(ids), 128), np.float32); model.eval()
    for i in range(0, len(ids), 256):
        ch = ids[i:i+256]; x = torch.stack([encode_pad(look[d]) for d in ch]).to(device)
        out[i:i+len(ch)] = model.encoder(x).cpu().numpy()
    return out

def esm_pool(feed):
    look = LOOK[feed]; ids = POOL_IDS[feed]
    cf = f'{CACHE}/esm2_28_{feed}.npy'; sf = f'{CACHE}/esm2_28_{feed}.meta.json'
    if os.path.exists(cf) and os.path.exists(sf):
        meta = json.load(open(sf))
        if meta.get('model') == ESM2_MODEL and meta.get('ids') == ids:
            return np.load(cf)
        print(f'  [{feed}] cache mismatch -> recompute')
    e = esm_embed([look[d] for d in ids]); np.save(cf, e)
    json.dump({'model': ESM2_MODEL, 'ids': ids}, open(sf, 'w')); return e

SNN_POOL = {f: snn_pool(f) for f in FEEDS}; print('SNN pool embeddings done.')
ESM_POOL = {f: esm_pool(f) for f in FEEDS}; print('ESM2 pool embeddings done.')
POS = {f: {d: i for i, d in enumerate(POOL_IDS[f])} for f in FEEDS}

## 6. Oracle (exact-Levenshtein neighbour sets) + metric machinery (AUROC + MAP@10)

In [ ]:
def build_oracle(feed):
    lk = LOOK[feed]; ids = POOL_IDS[feed]; idx = {d: i for i, d in enumerate(ids)}
    seqs = [lk[d] for d in ids]; lens = np.array([len(s) for s in seqs]); q_ids = QUERIES[feed]
    if not q_ids: return dict(idx=idx, q_ids=[], true_sets={})
    D = rf_cdist([lk[q] for q in q_ids], seqs, scorer=RFLev.distance, workers=-1).astype(float)
    ts = {}
    for r, q in enumerate(q_ids):
        qi = idx[q]; den = np.maximum(lens, lens[qi]); den[den == 0] = 1
        osim = 1.0 - D[r]/den; osim[qi] = -np.inf; ts[q] = set(np.where(osim >= BAND_HIGH)[0].tolist())
    print(f'  oracle[{feed}]: {len(q_ids)} queries, median|T|={int(np.median([len(ts[q]) for q in q_ids]))}')
    return dict(idx=idx, q_ids=q_ids, true_sets=ts)
print('Building exact-Levenshtein oracle...'); ORACLE = {f: build_oracle(f) for f in FEEDS}

def auroc_free(feed, emb, is_snn):
    e = EVAL[feed]; pos = POS[feed]
    a = np.array([pos[d] for d in e.domain_a]); b = np.array([pos[d] for d in e.domain_b])
    sim = (1.0 - np.linalg.norm(emb[a]-emb[b], axis=1)/2.0) if is_snn else np.sum(emb[a]*emb[b], axis=1)
    y = e.is_high.values
    return float(roc_auc_score(y, sim)) if 0 < y.sum() < len(y) else np.nan

def _ap(order, ts, k):
    nt = len(ts);  hits = 0; ap = 0.0
    if nt == 0: return np.nan
    for i, oi in enumerate(order[:k], 1):
        if oi in ts: hits += 1; ap += hits/i
    return ap/min(nt, k)

def retrieval_metrics(feed, emb, is_snn, n_boot=2000):
    cache = ORACLE[feed]; idx = cache['idx']; ap=[]; r10=[]; r50=[]; h10=[]; nt=[]
    for q in cache['q_ids']:
        qi = idx[q]; ts = cache['true_sets'][q]
        if not ts: continue
        sim = (1.0 - np.linalg.norm(emb-emb[qi], axis=1)/2.0) if is_snn else emb @ emb[qi]
        sim[qi] = -np.inf; order = np.argsort(-sim); nt.append(len(ts))
        ap.append(_ap(order, ts, 10))
        r10.append(len(set(order[:10].tolist()) & ts)/len(ts)); r50.append(len(set(order[:50].tolist()) & ts)/len(ts))
        h10.append(1.0 if len(set(order[:10].tolist()) & ts) else 0.0)
    ap = np.array(ap); n = len(ap)
    out = {'MAP@10': float(ap.mean()) if n else np.nan, 'recall@10': float(np.mean(r10)) if n else np.nan,
           'recall@50': float(np.mean(r50)) if n else np.nan, 'hit@10': float(np.mean(h10)) if n else np.nan,
           'median_n_true': float(np.median(nt)) if n else np.nan, 'n_q': n}
    if n >= 2:
        bi = np.random.default_rng(SEED).integers(0, n, (n_boot, n)); b = ap[bi].mean(1)
        out['MAP@10_lo'], out['MAP@10_hi'] = float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))
    else:
        out['MAP@10_lo'] = out['MAP@10_hi'] = np.nan
    return out

## 7. Quality table — AUROC + MAP@10 (SNN vs ESM2 vs length baseline)

In [ ]:
rows = []
for feed in FEEDS:
    au_s = auroc_free(feed, SNN_POOL[feed], True)
    au_e = auroc_free(feed, ESM_POOL[feed], False)
    rt_s = retrieval_metrics(feed, SNN_POOL[feed], True)
    rt_e = retrieval_metrics(feed, ESM_POOL[feed], False)
    plen = np.array([len(LOOK[feed][d]) for d in POOL_IDS[feed]], float)
    # length-only "embedding": rank by 1 - |dlen|/max(len); reuse retrieval_metrics via a 1-D trick
    cache = ORACLE[feed]; ap=[];
    for q in cache['q_ids']:
        qi = cache['idx'][q]; ts = cache['true_sets'][q]
        if not ts: continue
        sim = 1.0 - np.abs(plen - plen[qi])/np.maximum(plen, plen[qi]); sim[qi] = -np.inf
        ap.append(_ap(np.argsort(-sim), ts, 10))
    len_map = float(np.mean(ap)) if ap else np.nan
    rows.append(dict(feed=feed, n_pos=int(feed_high(feed).shape[0]), n_q=rt_s['n_q'], med_T=rt_s['median_n_true'],
                     auroc_snn=au_s, auroc_esm=au_e,
                     map_snn=rt_s['MAP@10'], map_snn_lo=rt_s['MAP@10_lo'], map_snn_hi=rt_s['MAP@10_hi'],
                     map_esm=rt_e['MAP@10'], map_esm_lo=rt_e['MAP@10_lo'], map_esm_hi=rt_e['MAP@10_hi'],
                     map_len=len_map, rec50_snn=rt_s['recall@50'], rec50_esm=rt_e['recall@50']))
Q = pd.DataFrame(rows)
print('=' * 112)
print(f'QUALITY — AUROC (pairwise is-high) + MAP@10 (full-pool, exact-Lev oracle).  ESM2={ESM2_MODEL}')
print('=' * 112)
print(f'{"feed":<5}{"n_pos":>6}{"n_q":>5}{"med|T|":>7} | {"AUROC SNN":>10}{"AUROC ESM2":>11} | '
      f'{"MAP@10 SNN":>11}{"MAP@10 ESM2":>12}{"MAP@10 len":>11}')
print('-' * 112)
for _, r in Q.iterrows():
    print(f'{r.feed:<5}{r.n_pos:>6}{r.n_q:>5}{r.med_T:>7.0f} | {r.auroc_snn:>10.3f}{r.auroc_esm:>11.3f} | '
          f'{r.map_snn:>11.3f}{r.map_esm:>12.3f}{r.map_len:>11.3f}')
Q.to_csv('colab28_quality.csv', index=False)
print('\nMAP ceiling = 1.0 by construction (relevance = the metric the encoder approximates).')
print('AA n_pos=5 -> underpowered; SS/3Di powered. AUROC = pairwise (easy); MAP@10 = retrieval (hard).')

## 8. Figure — AUROC and MAP@10 side by side (SNN vs ESM2)

In [ ]:
import matplotlib.pyplot as plt
x = np.arange(len(FEEDS)); w = 0.26
fig, ax = plt.subplots(1, 2, figsize=(14, 5.2))
# AUROC
ax[0].bar(x-w/2, Q.auroc_snn, w, color='#1f77b4', label='SNN-free')
ax[0].bar(x+w/2, Q.auroc_esm, w, color='#ff7f0e', label='ESM2-free')
ax[0].axhline(0.5, ls='--', color='grey'); ax[0].set_ylim(0, 1.05)
ax[0].set_title('AUROC is-high (pairwise discrimination)'); ax[0].set_xticks(x); ax[0].set_xticklabels(FEEDS)
ax[0].legend()
for i, r in Q.iterrows():
    ax[0].annotate(f'n_pos={r.n_pos}', (i, 0), (0, -26), textcoords='offset points', ha='center',
                   va='top', fontsize=8, color='dimgray')
# MAP@10 with CI + length baseline
err_s = np.vstack([Q.map_snn - Q.map_snn_lo, Q.map_snn_hi - Q.map_snn])
err_e = np.vstack([Q.map_esm - Q.map_esm_lo, Q.map_esm_hi - Q.map_esm])
ax[1].bar(x-w, Q.map_snn, w, color='#1f77b4', label='SNN-free'); ax[1].errorbar(x-w, Q.map_snn, yerr=err_s, fmt='none', ecolor='k', capsize=3)
ax[1].bar(x,   Q.map_esm, w, color='#ff7f0e', label='ESM2-free'); ax[1].errorbar(x, Q.map_esm, yerr=err_e, fmt='none', ecolor='k', capsize=3)
ax[1].bar(x+w, Q.map_len, w, color='#888888', label='length-only')
ax[1].set_ylim(0, 1.05); ax[1].set_title('MAP@10 (full-pool retrieval vs exact-Lev oracle; ceiling 1.0)')
ax[1].set_xticks(x); ax[1].set_xticklabels(FEEDS); ax[1].legend()
for i, r in Q.iterrows():
    ax[1].annotate(f'med|T|={r.med_T:.0f}', (i, 0), (0, -26), textcoords='offset points', ha='center',
                   va='top', fontsize=8, color='dimgray')
plt.tight_layout(); plt.savefig('colab28_quality.png', dpi=150, bbox_inches='tight'); plt.show()

## 9. Runtime — three parts: construction, inference, all-together

- **Construction (one-time, amortized):** SNN = data-gen + train + pool-encode; ESM2 = model load +
  pool-encode; exact Levenshtein = **0** (no index).
- **Inference (per query):** encode the query + search the pool. Exact Levenshtein = brute-force
  `1 query × N` Levenshtein.
- **All-together:** end-to-end for `Q` queries = construction + `Q ×` inference.

Encode runs on `{device}`; vector search + Levenshtein on CPU (matching the colab16 7×-CPU framing).
Pool = the AA feed (~10.5k). Warmup + repeats; min taken.

In [ ]:
TIME_FEED = 'AA'; ids = POOL_IDS[TIME_FEED]; look = LOOK[TIME_FEED]
pool_strs = [look[d] for d in ids]; POOLN = len(ids)
snn_pool_cpu = SNN_POOL[TIME_FEED].astype('float32'); esm_pool_cpu = ESM_POOL[TIME_FEED].astype('float32')
q_strs = [look[d] for d in QUERIES[TIME_FEED][:8]] or pool_strs[:8]

@torch.no_grad()
def snn_encode(seqs):
    x = torch.stack([encode_pad(s) for s in seqs]).to(device); return model_aa.encoder(x).cpu().numpy()
def _rep(fn, reps=5):
    fn();  # warmup
    if device.type == 'cuda': torch.cuda.synchronize()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn()
        if device.type == 'cuda': torch.cuda.synchronize()
        ts.append(time.perf_counter()-t0)
    return float(np.min(ts))

# --- construction: pool encode (one-time) ---
snn_poolenc_s = _rep(lambda: snn_encode(pool_strs[:2000]), reps=2) * (POOLN/2000)
esm_poolenc_s = _rep(lambda: esm_embed(pool_strs[:512]), reps=1) * (POOLN/512)
SNN_CONSTRUCT = AA_DATAGEN_S + AA_TRAIN_S + snn_poolenc_s
ESM_CONSTRUCT = ESM_LOAD_S + esm_poolenc_s
LEV_CONSTRUCT = 0.0

# --- inference per query: encode 1 + search N ---
snn_qenc = _rep(lambda: snn_encode(q_strs[:1]))
esm_qenc = _rep(lambda: esm_embed(q_strs[:1]))
qv_s = snn_pool_cpu[0]; qv_e = esm_pool_cpu[0]
snn_search = _rep(lambda: np.argpartition(-(snn_pool_cpu @ qv_s), 10)[:10])
esm_search = _rep(lambda: np.argpartition(-(esm_pool_cpu @ qv_e), 10)[:10])
lev_query = _rep(lambda: rf_cdist([q_strs[0]], pool_strs, scorer=RFLev.distance, workers=-1), reps=3)
SNN_QUERY = snn_qenc + snn_search; ESM_QUERY = esm_qenc + esm_search; LEV_QUERY = lev_query

print('=' * 86)
print(f'RUNTIME  (encode on {device.type.upper()}; search+Lev on CPU; pool N={POOLN:,})')
print('=' * 86)
print(f'{"":<34}{"SNN":>16}{"ESM2":>16}{"exact Lev":>16}')
print('-' * 86)
print(f'{"params":<34}{snn_params:>16,}{esm_params:>16,}{"-":>16}')
print(f'{"CONSTRUCT total (one-time)":<34}{SNN_CONSTRUCT:>15.1f}s{ESM_CONSTRUCT:>15.1f}s{LEV_CONSTRUCT:>15.1f}s')
print(f'{"  - train/load":<34}{AA_DATAGEN_S+AA_TRAIN_S:>15.1f}s{ESM_LOAD_S:>15.1f}s{"-":>16}')
print(f'{"  - pool encode":<34}{snn_poolenc_s:>15.1f}s{esm_poolenc_s:>15.1f}s{"-":>16}')
print(f'{"INFER per query (ms)":<34}{SNN_QUERY*1e3:>16.2f}{ESM_QUERY*1e3:>16.2f}{LEV_QUERY*1e3:>16.2f}')
print(f'{"  - encode query (ms)":<34}{snn_qenc*1e3:>16.2f}{esm_qenc*1e3:>16.2f}{"-":>16}')
print(f'{"  - search pool (ms)":<34}{snn_search*1e3:>16.3f}{esm_search*1e3:>16.3f}{LEV_QUERY*1e3:>16.2f}')
print('-' * 86)
print(f'  encode-throughput ratio ESM2/SNN = {esm_qenc/snn_qenc:.0f}x   | param ratio = {esm_params/snn_params:.0f}x')
timing = dict(device=device.type, poolN=POOLN, snn_params=int(snn_params), esm_params=int(esm_params),
              SNN_CONSTRUCT=SNN_CONSTRUCT, ESM_CONSTRUCT=ESM_CONSTRUCT,
              SNN_QUERY=SNN_QUERY, ESM_QUERY=ESM_QUERY, LEV_QUERY=LEV_QUERY)
json.dump(timing, open('colab28_timing.json', 'w'), indent=2, default=float)

## 10. All-together — end-to-end time for Q queries (the amortization crossover)

In [ ]:
Qs = np.array([1, 10, 100, 1_000, 10_000, 100_000, 1_000_000])
tot_snn = SNN_CONSTRUCT + Qs*SNN_QUERY
tot_esm = ESM_CONSTRUCT + Qs*ESM_QUERY
tot_lev = LEV_CONSTRUCT + Qs*LEV_QUERY
fig, ax = plt.subplots(figsize=(8.6, 5.8))
ax.plot(Qs, tot_lev, 'o-', color='#d62728', lw=2, label='exact Levenshtein (no construction)')
ax.plot(Qs, tot_snn, 's-', color='#1f77b4', lw=2, label='SNN (construct + search)')
ax.plot(Qs, tot_esm, '^-', color='#ff7f0e', lw=2, label='ESM2 (construct + search)')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('number of queries Q (against the same pool)'); ax.set_ylabel('total wall-clock time (s)')
ax.set_title(f'End-to-end cost vs #queries (pool N={POOLN:,}, encode on {device.type.upper()})')
# crossover: smallest Q where SNN total < Lev total
cross = Qs[np.where(tot_snn < tot_lev)[0]]
if len(cross):
    ax.axvline(cross[0], ls=':', color='grey'); ax.text(cross[0], ax.get_ylim()[0]*1.5,
               f' SNN beats Lev\n from Q≈{cross[0]:,}', fontsize=9, color='grey')
ax.grid(True, which='both', alpha=0.3); ax.legend(loc='upper left', framealpha=0.9)
plt.tight_layout(); plt.savefig('colab28_amortization.png', dpi=150, bbox_inches='tight'); plt.show()
print('one-off query (Q=1): exact Lev wins (no construction).  Many queries: embedding amortizes.')
print(f'SNN construction ({SNN_CONSTRUCT:.0f}s) << ESM2 construction ({ESM_CONSTRUCT:.0f}s) -> SNN below ESM2 for all Q.')

## How to read this notebook

**Quality (§7–8).** Two metrics, two questions:
- **AUROC** (pairwise: high vs random) is high for both — an *easy* discrimination; AA is n_pos=5
  (underpowered, read SS/3Di). SNN ≥ ESM2 here (its geometry is edit-distance-aligned).
- **MAP@10** (full-pool retrieval vs the exact-Lev oracle) is the *hard* deployment metric. Expect
  **SNN ≈ ESM2, ESM2 marginally ahead** on SS/3Di (the colab18 finding) — **not** an SNN retrieval win.
  Both bounded by the length-only baseline check + the crowded `med|T|` in SS/3Di.

**Runtime (§9–10).** The SNN's real case:
- **Construction:** SNN (data-gen + train a 150k-param net + encode the pool) is far cheaper than
  ESM2 (load a 34M-param transformer + encode the pool with it). Exact Lev builds nothing.
- **Inference:** the *search* is the same vector op for SNN and ESM2; the difference is **encode the
  query** — ESM2 is ~the param-ratio slower. Exact Lev pays a full `N`-candidate scan every query.
- **All-together:** for one-off queries exact Levenshtein wins (no construction); past the crossover
  Q the embedding amortizes, and the **SNN sits below ESM2 for every Q** because its construction is
  cheap. This is the efficiency leg of the thesis case.

**Honest headline:** the SNN **matches a 227×-larger pretrained LM at retrieval while being zero-shot,
a pure edit-distance proxy, and far cheaper to build and run** — it wins on pairwise discrimination +
efficiency, *not* on retrieval. ESM2-35M is conservative; a larger ESM2 would likely widen its
retrieval edge and its runtime disadvantage. No biological claim.